In [21]:
import pandas as pd
import numpy as np

COLUMNS = ["city", "price", "location", "beds", "baths", "size", "title"]

combined = pd.read_csv("../data/all_listings_raw.csv", names=COLUMNS)

print("Shape:", combined.shape)
print("\nFirst 5 rows:")
combined.head()

Shape: (7500, 7)

First 5 rows:


,city,price,location,beds,baths,size,title
0,islamabad,5.75 Crore,"Pakistan Town - Phase 1, Pakistan Town",4.0,6.0,1.2 Kanal,House Is Available For sale In Pakistan Town -...
1,islamabad,4.1 Crore,"Top City 1, Islamabad",5.0,6.0,10 Marla,10 Marla Brand New House For Sale
2,islamabad,11.7 Crore,"DHA Defence Phase 2, DHA Defence",5.0,6.0,1 Kanal,DHA Phase 2 Luxury House For Sale Beautiful Lo...
3,islamabad,4 Crore,"Top City 1 - Block A, Top City 1",5.0,6.0,10 Marla,Affordable House For Sale In Top City 1 - Block A
4,islamabad,2.95 Crore,"Mumtaz City, Islamabad",5.0,6.0,8 Marla,Ready To sale A House 8 Marla In Mumtaz City M...


In [22]:
print("=== Unique Price Formats (sample) ===")
print(combined['price'].unique()[:30])

print("\n=== Unique Size Formats (sample) ===")
print(combined['size'].unique()[:30])

print("\n=== Missing Values ===")
print(combined.isnull().sum())

=== Unique Price Formats (sample) ===
<StringArray>
[ '5.75 Crore',   '4.1 Crore',  '11.7 Crore',     '4 Crore',  '2.95 Crore',
     '98 Lakh',     '48 Lakh',  '3.25 Crore',   '7.5 Crore',   '4.2 Crore',
  '1.95 Crore',  '11.9 Crore',   '1.9 Crore',   '2.4 Crore',   '2.7 Crore',
   '3.1 Crore',  '4.75 Crore',  '10.2 Crore',  '16.5 Crore',     '1 Crore',
   '52.8 Lakh',  '1.32 Crore',   '3.9 Crore',   '2.3 Crore',   '2.9 Crore',
     '55 Lakh',  '5.65 Crore',   '9.5 Crore',   '9.9 Crore', '13.75 Crore']
Length: 30, dtype: str

=== Unique Size Formats (sample) ===
<StringArray>
[ '1.2 Kanal',   '10 Marla',    '1 Kanal',    '8 Marla',    '5 Marla',
    '3 Marla',    '6 Marla',    '7 Marla',    '2 Kanal',  '3.8 Marla',
  '1.5 Marla',  '3.5 Marla',  '4.1 Marla',    '5 Kanal',   '11 Marla',
  '4.3 Marla',  '9.3 Marla',  '5.6 Marla',   '12 Marla',  '3.2 Marla',
  '1.7 Marla',    '4 Marla', '10.7 Marla',   '14 Marla',  '7.5 Marla',
  '2.1 Marla',  '1.9 Marla',  '5.5 Marla',  '2.5 Kanal',  '5.3

In [23]:
def parse_price(price_str):
    if pd.isnull(price_str):
        return None
    price_str = str(price_str).strip()
    
    # extract numeric part and unit
    if 'Arab' in price_str:
        return float(price_str.replace('Arab', '').strip()) * 1_000_000_000
    elif 'Crore' in price_str:
        return float(price_str.replace('Crore', '').strip()) * 10_000_000
    elif 'Lakh' in price_str:
        return float(price_str.replace('Lakh', '').strip()) * 100_000
    elif 'Thousand' in price_str:
        return float(price_str.replace('Thousand', '').strip()) * 1_000
    else:
        return None

combined['price_pkr'] = combined['price'].apply(parse_price)

# verify all formats
print("Price parsing verification:")
combined[['price', 'price_pkr']].drop_duplicates().dropna().sort_values('price_pkr', ascending=False).head(20)

Price parsing verification:


,price,price_pkr
2093,3.7 Arab,3.700000e+09
1331,3.5 Arab,3.500000e+09
5027,2 Arab,2.000000e+09
5678,1.95 Arab,1.950000e+09
1105,1.7 Arab,1.700000e+09
2195,1.62 Arab,1.620000e+09
545,1.55 Arab,1.550000e+09
535,1.5 Arab,1.500000e+09
56,1.4 Arab,1.400000e+09
6456,1.35 Arab,1.350000e+09


In [ ]:
# conversion factors
# 1 Kanal     = 20 Marla
# 1 Sq. Yd.   = 1/25.2929 Marla
# 1 Sq. Ft.   = 1/272.25 Marla

def parse_size(size_str):
    if pd.isnull(size_str):
        return None
    size_str = str(size_str).strip().replace(',', '')  # fix 1,000 -> 1000
    
    try:
        if 'Kanal' in size_str:
            return round(float(size_str.replace('Kanal', '').strip()) * 20, 2)
        elif 'Marla' in size_str:
            return round(float(size_str.replace('Marla', '').strip()), 2)
        elif 'Sq. Yd.' in size_str:
            return round(float(size_str.replace('Sq. Yd.', '').strip()) / 25.2929, 2)
        elif 'Sq. Ft.' in size_str:
            return round(float(size_str.replace('Sq. Ft.', '').strip()) / 272.25, 2)
        else:
            return None
    except ValueError:
        return None

combined['size_marla'] = combined['size'].apply(parse_size)


# verify all formats
print("Size parsing verification:")
combined[['size', 'size_marla']].drop_duplicates().dropna().sort_values('size_marla', ascending=False).head(10)

Null sizes after fix: 0
Size parsing verification:


,size,size_marla
2212,488 Kanal,9760.0
1602,88 Kanal,1760.0
2414,80 Kanal,1600.0
4204,64 Kanal,1280.0
702,40 Kanal,800.0
4419,24 Kanal,480.0
1351,23 Kanal,460.0
1408,20 Kanal,400.0
2676,16 Kanal,320.0
508,14 Kanal,280.0


In [29]:
def split_location(location, part):
    if pd.isnull(location):
        return None
    parts = location.split(',')
    if part == 'neighbourhood':
        return parts[0].strip()
    elif part == 'area':
        return parts[1].strip() if len(parts) > 1 else None

combined['neighbourhood'] = combined['location'].apply(lambda x: split_location(x, 'neighbourhood'))
combined['area']          = combined['location'].apply(lambda x: split_location(x, 'area'))

print("Location split verification:")
combined[['location', 'neighbourhood', 'area']].head(10)

Location split verification:


,location,neighbourhood,area
0,"Pakistan Town - Phase 1, Pakistan Town",Pakistan Town - Phase 1,Pakistan Town
1,"Top City 1, Islamabad",Top City 1,Islamabad
2,"DHA Defence Phase 2, DHA Defence",DHA Defence Phase 2,DHA Defence
3,"Top City 1 - Block A, Top City 1",Top City 1 - Block A,Top City 1
4,"Mumtaz City, Islamabad",Mumtaz City,Islamabad
5,"Alipur Farash, Islamabad",Alipur Farash,Islamabad
6,"Ali Pur, Islamabad",Ali Pur,Islamabad
7,"Soan Garden, Islamabad",Soan Garden,Islamabad
8,"Soan Garden - Block E, Soan Garden",Soan Garden - Block E,Soan Garden
9,"Soan Garden - Block H, Soan Garden",Soan Garden - Block H,Soan Garden


In [43]:
combined['price_per_marla'] = (combined['price_pkr'] / combined['size_marla']).round(2)

print("Price per Marla sample:")
combined[['city', 'neighbourhood', 'price', 'size_marla', 'price_per_marla']].sort_values('price_per_marla', ascending=False).head(30)

Price per Marla sample:


,city,neighbourhood,price,size_marla,price_per_marla
6580,karachi,Gulistan-e-Jauhar - Block 13,7.4 Crore,1.07,69158878.50
5963,karachi,Naya Nazimabad,6.3 Crore,1.07,58878504.67
932,islamabad,Giga Mall Extension Tower,4.2 Crore,0.80,52500000.00
2093,islamabad,F-7/3,3.7 Arab,80.00,46250000.00
1331,islamabad,F-7/3,3.5 Arab,80.00,43750000.00
2195,islamabad,E-7,1.62 Arab,50.00,32400000.00
102,islamabad,F-7,55 Crore,20.00,27500000.00
2097,islamabad,E-7,1.3 Arab,48.00,27083333.33
5380,karachi,Navy Housing Scheme Karsaz,50 Crore,19.77,25290844.71
6832,karachi,KDA Officers Society,24 Crore,9.49,25289778.71


In [33]:
# convert to numeric first in case they were read as strings
combined['beds']  = pd.to_numeric(combined['beds'],  errors='coerce')
combined['baths'] = pd.to_numeric(combined['baths'], errors='coerce')

# fill by size group first
combined['beds']  = combined.groupby('size_marla')['beds'].transform(
    lambda x: x.fillna(x.median())
)
combined['baths'] = combined.groupby('size_marla')['baths'].transform(
    lambda x: x.fillna(x.median())
)

# fallback to overall median
combined['beds']  = combined['beds'].fillna(combined['beds'].median())
combined['baths'] = combined['baths'].fillna(combined['baths'].median())

print("Missing after fix:")
print(combined[['beds', 'baths']].isnull().sum())

Missing after fix:
beds     0
baths    0
dtype: int64


In [34]:
before = len(combined)
df_clean = combined.dropna(subset=['price_pkr', 'size_marla', 'price_per_marla'])
after = len(combined)

print(f"Dropped {before - after} rows with unparseable price or size")
print(f"Remaining: {after} rows")

Dropped 0 rows with unparseable price or size
Remaining: 7500 rows


In [44]:
print("=== Shape ===")
print(combined.shape)

print("\n=== Data Types ===")
print(combined.dtypes)

print("\n=== Missing Values ===")
print(combined.isnull().sum())

print("\n=== Stats ===")
combined[['price_pkr', 'size_marla', 'price_per_marla', 'beds', 'baths']].describe()

=== Shape ===
(7500, 12)

=== Data Types ===
city                   str
price                  str
location               str
beds               float64
baths              float64
size                   str
title                  str
price_pkr          float64
size_marla         float64
neighbourhood          str
area                   str
price_per_marla    float64
dtype: object

=== Missing Values ===
city               0
price              0
location           0
beds               0
baths              0
size               0
title              0
price_pkr          0
size_marla         0
neighbourhood      0
area               0
price_per_marla    0
dtype: int64

=== Stats ===


,price_pkr,size_marla,price_per_marla,beds,baths
count,7.500000e+03,7500.000000,7.500000e+03,7500.000000,7500.000000
mean,8.415054e+07,16.106453,5.479419e+06,4.196333,4.575133
std,1.482203e+08,119.146660,3.452905e+06,1.718414,1.643111
min,1.800000e+06,0.800000,4.098361e+04,1.000000,1.000000
25%,2.250000e+07,5.000000,3.420789e+06,3.000000,3.000000
50%,4.300000e+07,9.880000,4.600000e+06,4.000000,5.000000
75%,8.642500e+07,19.770000,6.363636e+06,5.000000,6.000000
max,3.700000e+09,9760.000000,6.915888e+07,11.000000,7.000000
